# 08 — Definition Comparison & Final Selection

Compare D1, D2, D3, D4 head-to-head on the test set.
Select the winner using the OOS Sharpe. The test set is used **once** here — not before.

Selection rule: Definition with highest net OOS Sharpe that passes all minimum bars.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from src.plots import plot_definition_comparison, plot_signal_count_vs_sharpe

proc_dir = Path('../data/processed')

results = pd.read_parquet(proc_dir / 'backtest_results.parquet')
test_results = results[results['split'] == 'test'].copy()
val_results  = results[results['split'] == 'val'].copy()

print('Test results loaded:')
print(test_results[['definition', 'best_H', 'sharpe', 'max_drawdown_pct', 'n_trades', 'cagr']].to_string(index=False))

## Head-to-Head Comparison Table

In [ ]:
METRICS = ['sharpe', 'sortino', 'max_drawdown_pct', 'cagr', 'win_rate_pct', 'n_trades', 'cost_drag_pct']
available_metrics = [m for m in METRICS if m in test_results.columns]

comparison = test_results.set_index('definition')[available_metrics]
print('=== FINAL COMPARISON TABLE (Test Set) ===')
print(comparison.to_string())

# Minimum bars check
comparison['passes_sharpe']   = comparison['sharpe'] >= 1.0
comparison['passes_drawdown'] = comparison['max_drawdown_pct'] < 20.0
comparison['passes_trades']   = comparison['n_trades'] >= 30
comparison['all_bars_passed'] = (
    comparison['passes_sharpe'] &
    comparison['passes_drawdown'] &
    comparison['passes_trades']
)

print('\n=== Minimum Bars Check ===')
print(comparison[['passes_sharpe', 'passes_drawdown', 'passes_trades', 'all_bars_passed']].to_string())

## OOS/IS Ratio (Overfitting Check)

In [ ]:
# Get best validation Sharpe per definition (IS proxy) and test Sharpe (OOS)
is_sharpe = val_results.groupby('definition')['sharpe'].max()
oos_sharpe = test_results.set_index('definition')['sharpe']

ratio = (oos_sharpe / is_sharpe).rename('oos_is_ratio')
print('=== OOS/IS Sharpe Ratio (must be > 0.5) ===')
for defn, r in ratio.items():
    status = 'PASS' if r >= 0.5 else 'FAIL'
    print(f'  [{status}] {defn}: IS={is_sharpe.get(defn, float("nan")):.3f}, '
          f'OOS={oos_sharpe.get(defn, float("nan")):.3f}, ratio={r:.3f}')

## Select Winner

In [ ]:
viable = comparison[comparison['all_bars_passed']]

if viable.empty:
    print('NO definition passes all minimum bars on the test set.')
    print('Do not deploy any strategy. Go back to hypothesis stage.')
    winner = None
else:
    # Select by highest OOS Sharpe among those that also pass OOS/IS ratio check
    viable_with_ratio = viable.join(ratio, how='left')
    viable_final = viable_with_ratio[viable_with_ratio.get('oos_is_ratio', 0.5) >= 0.5]

    if viable_final.empty:
        viable_final = viable  # relax OOS/IS if all fail it

    winner = viable_final['sharpe'].idxmax()
    best_H = test_results.set_index('definition').loc[winner, 'best_H']

    print(f'=== WINNER ===')
    print(f'Definition: {winner}')
    print(f'Holding period: {int(best_H)} minutes')
    print()
    print(comparison.loc[winner, available_metrics].to_string())

## Comparison Plots

In [ ]:
if not test_results.empty:
    comp_table = comparison.reset_index()
    comp_table.index = comp_table['definition']

    # Side-by-side Sharpe
    fig = plot_definition_comparison(comp_table, metric='sharpe',
                                      title='Sharpe Ratio: D1 vs D2 vs D3 vs D4 (Test Set)')
    plt.show()
    plt.close()

    # Max drawdown comparison
    fig = plot_definition_comparison(comp_table, metric='max_drawdown_pct',
                                      title='Max Drawdown % by Definition (lower = better)')
    plt.show()
    plt.close()

    # Signal count vs Sharpe scatter
    fig = plot_signal_count_vs_sharpe(comp_table)
    plt.show()
    plt.close()

## Parameter Sensitivity (Winner Only)

In [ ]:
if winner is not None:
    from src.signal import compute_all_signals
    from src.backtest import run_backtest, train_val_test_split
    from src.metrics import sharpe as compute_sharpe
    import yaml

    with open('../config.yaml') as f:
        cfg = yaml.safe_load(f)

    raw_dir = Path('../data/raw')
    btc = pd.read_parquet(raw_dir / 'crypto' / 'crypto_1m_BTCUSDT.parquet')
    btc_ret = btc.set_index('timestamp_utc').sort_index()['log_ret']

    final_markets = pd.read_parquet(raw_dir / 'kalshi' / 'final_market_list.parquet')

    # Perturb the primary parameter ±20% and report Sharpe
    base_cfg = dict(cfg)
    perturbations = [0.8, 0.9, 1.0, 1.1, 1.2]  # multipliers
    param_name = 'd1_z_threshold' if winner == 'D1' else 'd2_abs_threshold' if winner == 'D2' else 'd3_rel_threshold'
    base_val = cfg['signal'].get(param_name, 2.0)

    print(f'Sensitivity: {param_name} = {base_val} ± 20%')
    sensitivity_rows = []

    for mult in perturbations:
        perturbed_cfg = dict(cfg)
        perturbed_cfg['signal'] = dict(cfg['signal'])
        perturbed_cfg['signal'][param_name] = base_val * mult

        # Rebuild signals
        all_sigs = {}
        for ticker in final_markets['ticker']:
            path = raw_dir / 'kalshi' / f'{ticker}.parquet'
            if not path.exists():
                continue
            raw_df = pd.read_parquet(path).set_index('timestamp_utc').sort_index()
            if 'prob_mid' not in raw_df.columns:
                continue
            prob_mid = raw_df['prob_mid'].dropna()
            volume = pd.to_numeric(raw_df.get('volume', pd.Series(dtype=float)), errors='coerce').reindex(prob_mid.index)
            sigs = compute_all_signals(prob_mid, volume=volume, cfg=perturbed_cfg)
            if winner in sigs:
                all_sigs[ticker] = sigs[winner]['signal']

        if not all_sigs:
            continue
        sig_df = pd.DataFrame(all_sigs)
        combined = sig_df.stack()
        combined = combined[combined != 0]
        if combined.empty:
            continue
        flat = combined.groupby(level=0).apply(lambda x: x.iloc[x.abs().argmax()])
        flat = flat.reindex(btc_ret.index).fillna(0)

        _, _, test_sig = train_val_test_split(flat)
        bt = run_backtest(test_sig, btc_ret, holding_period=int(best_H),
                          commission_rt=cfg['backtest']['commission_rt'],
                          slippage=cfg['backtest']['slippage'])
        sh = compute_sharpe(bt['net_ret'].dropna())
        sensitivity_rows.append({'multiplier': mult, 'param_value': round(base_val * mult, 4), 'sharpe': round(sh, 3)})
        print(f'  {mult:.1f}x ({base_val * mult:.4f}): Sharpe = {sh:.3f}')

    sens_df = pd.DataFrame(sensitivity_rows)
    robust = (sens_df['sharpe'] >= 1.0).all()
    print(f'\nRobust to ±20% parameter perturbation: {robust}')
    if not robust:
        print('WARNING: Strategy is sensitive to parameter choice — treat results with caution.')

## Final Summary

In [ ]:
print('=== FINAL RESEARCH SUMMARY ===')
print()
print(f'Winner definition: {winner}')
if winner is not None:
    row = test_results.set_index('definition').loc[winner]
    print(f'Holding period: {int(row["best_H"])} minutes')
    print(f'Net Sharpe (OOS): {row["sharpe"]:.3f}')
    print(f'Max Drawdown: {row["max_drawdown_pct"]:.1f}%')
    print(f'CAGR: {row["cagr"]:.1f}%')
    print(f'Trades in test set: {int(row["n_trades"])}')
    print()
    print('Next steps if deploying:')
    print('  1. Set up Kalshi WebSocket feed for real-time probability monitoring')
    print('  2. Connect to Binance/Coinbase API for order execution')
    print(f'  3. Apply {winner} jump definition with cooldown={cfg["signal"]["cooldown_minutes"]}min')
    print('  4. Use 1% position size per trade, stop-loss at 1.5x pre-event vol')
    print('  5. Monitor: track live Sharpe vs backtest Sharpe weekly')